# 02 · Baselines (Colab)

Phase 2 of `notes/plan.md` (week 3 design, week 4 performance). Four baselines, scored with the same metrics on `splits/splits.csv`:

| | Baseline | Learns from |
|---|---|---|
| B0 | Training-set mean count | nothing |
| B1 | Per-session mean count (the "recognize the session" shortcut) | session label |
| B4 | Cellpose-SAM zero-shot + blue-colour rule for Live/Dead | one colour threshold |
| B7 | YOLO26n fine-tuned with default settings | our boxes |

Two test levels: the **random split** (`split == "test"`, 75 images) and **leave-one-session-out** (train on 4 sessions, test on the 5th). Results go to `outputs/results/baselines.csv`, figures to `outputs/figures/baselines/`.

Run `00_setup_and_download.ipynb` first, and use a GPU runtime. Cellpose masks are cached on Drive, and YOLO training is skipped when its weights already exist there, so a disconnected run picks up where it stopped. Upload your local `outputs/` to Drive to skip both entirely.

In [ ]:
# --- Colab bootstrap (identical in every notebook in this folder, except EXTRAS) ---
# Expects the contents of notebooks/notebooks_colab_draft/ uploaded to DRIVE_ROOT on Google Drive:
# pyproject.toml, src/cellcount/, splits/splits.csv and these notebooks (see README.md).
# data/ and outputs/ are created there too, so they survive runtime resets.
# Secrets (key icon in the left sidebar, notebook access on): ROBOFLOW_EXPORT_URL.
import os, shutil, subprocess, sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/cell-image-counting"
EXTRAS = "[model]"  # "[model]" adds torch, ultralytics and cellpose

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    root = Path(DRIVE_ROOT)
    missing = [p for p in ("pyproject.toml", "src/cellcount/__init__.py", "splits/splits.csv") if not (root / p).exists()]
    if missing:
        raise FileNotFoundError(f"missing in {root}: {missing}. Upload notebooks_colab_draft/ there (README.md).")
    # Install from a local copy: Drive is slow and pip would write build files into it.
    # Re-copied on every run, so re-uploaded code is picked up after a runtime restart.
    code_dir = Path("/content/cellcount_code")
    shutil.rmtree(code_dir, ignore_errors=True)
    shutil.copytree(root / "src", code_dir / "src", ignore=shutil.ignore_patterns("__pycache__", "*.egg-info"))
    shutil.copy(root / "pyproject.toml", code_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{code_dir}{EXTRAS}"], check=True)
    sys.path.insert(0, str(code_dir / "src"))  # import without restarting the runtime
    os.environ["CELLCOUNT_STORAGE"] = DRIVE_ROOT  # data/, outputs/ and splits/ live here
    os.chdir("/content")  # pretrained weights (yolo26n.pt, ...) download here, not to Drive

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: Runtime > Change runtime type > T4 GPU (or better). CPU training takes many hours.")

In [ ]:
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from cellcount import baselines, cellpose_zs, viz, yolo
from cellcount.config import get_paths
from cellcount.data import RAW_DIRNAME, load_coco
from cellcount.eda import count_bins
from cellcount.metrics import count_metrics, count_metrics_by, detection_metrics, per_image_counts
from cellcount.splits import leave_one_source_out, load_splits

sns.set_theme(style="whitegrid", context="notebook")
paths = get_paths()
FIG = paths.outputs / "figures" / "baselines"
FIG.mkdir(parents=True, exist_ok=True)
(paths.outputs / "results").mkdir(exist_ok=True)

# Copy the images to the runtime's local disk: Drive reads are slow, and export_yolo makes
# symlinks, which Drive doesn't support. Results and weights still go to Drive (paths.outputs).
WORK = Path("/content/work") if IN_COLAB else paths.data
if not (WORK / RAW_DIRNAME).exists():
    shutil.copytree(paths.data / RAW_DIRNAME, WORK / RAW_DIRNAME)
images, boxes = load_coco(WORK / RAW_DIRNAME)
split_df = load_splits(paths.root / "splits" / "splits.csv")
truth = per_image_counts(boxes, images["uid"]).rename(columns={"live": "live_true", "dead": "dead_true"})
df = (images[["uid", "path", "width", "height", "source"]]
      .merge(split_df[["uid", "split"]], on="uid")
      .merge(truth, on="uid"))
df["total_true"] = df["live_true"] + df["dead_true"]
df["count_bin"] = count_bins(df["total_true"])
train, valid, test = (df[df["split"] == s] for s in ("train", "valid", "test"))
print(df["split"].value_counts().to_dict(), "| sessions:", df["source"].nunique())

In [ ]:
results = []   # one row per method x test level x fold
preds = {}     # per-image predictions on the random-split test, for the figures


def with_counts(frame, pred_boxes):
    """Attach live_pred / dead_pred from predicted boxes to a per-image frame."""
    c = per_image_counts(pred_boxes, frame["uid"]).rename(columns={"live": "live_pred", "dead": "dead_pred"})
    return frame.merge(c, on="uid")


def evaluate(method, level, fold, counted, pred_boxes=None):
    row = {"method": method, "level": level, "fold": fold, **count_metrics(counted)}
    if pred_boxes is not None:
        det = detection_metrics(pred_boxes[pred_boxes["uid"].isin(counted["uid"])],
                                boxes[boxes["uid"].isin(counted["uid"])])
        for cls, m in det.iterrows():
            row.update({f"{k}_{cls.lower()}": m[k] for k in ("precision", "recall", "f1")})
    results.append(row)
    return row

## B0, B1: reference baselines

On the random split, B1 shows how much the session alone explains. Under leave-one-session-out the held-out session is unseen, so B1 falls back to B0.

In [ ]:
for name, fn in [("B0 mean", baselines.predict_mean), ("B1 session mean", baselines.predict_session_mean)]:
    preds[name] = fn(train, test)
    evaluate(name, "random split", "test", preds[name])
    loso = pd.concat([fn(tr, te) for _, tr, te in leave_one_source_out(df)])
    evaluate(name, "leave-one-session-out", "pooled", loso)
pd.DataFrame(results).set_index(["method", "level"])[["mae_total", "mae_dead", "r2_total", "within_10pct"]].round(3)

## B4: Cellpose-SAM zero-shot + colour rule

Cellpose (`cpsam_v2`) segments every cell without any training on our data. Default thresholds miss most pale live cells, so `cellpose_zs.EVAL_KWARGS` loosens them. Each cell is then called **Dead** when its blueness `B − (R+G)/2`, relative to the image background, exceeds a threshold. That threshold and a minimum cell area (to drop debris) are fitted on the training images only, to minimize count errors. On the labelled boxes this blueness separates the classes with AUC ≈ 0.99.

In [ ]:
masks = cellpose_zs.segment(df, paths.outputs / "cache" / "cellpose")
cells = cellpose_zs.masks_to_cells(masks, df)
print(f"{len(cells):,} segmented cells vs {len(boxes):,} labelled boxes")

min_area, dead_t = cellpose_zs.fit_rule(cells, train)
cp_boxes = cellpose_zs.classify(cells, dead_t, min_area)
print(f"fitted on train: min area = {min_area:.0f} px, Dead when blueness above background > {dead_t:.1f}")

preds["B4 Cellpose-SAM"] = with_counts(test, cp_boxes)
evaluate("B4 Cellpose-SAM", "random split", "test", preds["B4 Cellpose-SAM"], cp_boxes)

loso = []
for source, tr, te in leave_one_source_out(df):
    a, t = cellpose_zs.fit_rule(cells, tr)
    fold_boxes = cellpose_zs.classify(cells[cells["uid"].isin(te["uid"])], t, a)
    counted = with_counts(te, fold_boxes)
    evaluate("B4 Cellpose-SAM", "leave-one-session-out", source, counted, fold_boxes)
    loso.append((counted, fold_boxes))
evaluate("B4 Cellpose-SAM", "leave-one-session-out", "pooled",
         pd.concat([c for c, _ in loso]), pd.concat([b for _, b in loso]))

## B7: YOLO26n, fine-tuned with default settings

Exports our split to YOLO format (images are symlinks), fine-tunes the COCO-pretrained nano model on the GPU, then picks the confidence threshold that minimizes total-count MAE on **valid**. Training takes roughly 15–30 min on a T4 and is skipped when the weights exist on Drive.

In [ ]:
data_yaml = yolo.export_yolo(images, boxes, split_df, WORK / "yolo")
weights = paths.outputs / "yolo" / "yolo26n_default" / "weights" / "best.pt"
if not weights.exists():
    weights = yolo.train(data_yaml, paths.outputs / "yolo", model="yolo26n.pt", name="yolo26n_default")

yolo_boxes = yolo.predict(weights, pd.concat([valid, test]))
conf_t = yolo.tune_conf(yolo_boxes[yolo_boxes["uid"].isin(valid["uid"])], valid)
yolo_kept = yolo_boxes[yolo_boxes["conf"] >= conf_t]
print(f"confidence threshold (tuned on valid) = {conf_t:.2f}")

preds["B7 YOLO26n"] = with_counts(test, yolo_kept)
evaluate("B7 YOLO26n", "random split", "test", preds["B7 YOLO26n"], yolo_kept)

Training curves from Ultralytics (loss and mAP per epoch on valid):

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(weights.parents[1] / "results.png"), width=900))

### Optional: YOLO leave-one-session-out

Five more trainings (roughly 1.5–2.5 h on a T4). Off by default; Phase 4 covers it.

In [ ]:
RUN_YOLO_LOSO = False

if RUN_YOLO_LOSO:
    loso = []
    for source, tr, te in leave_one_source_out(df):
        fold_split = split_df.assign(split=np.where(split_df["source"] == source, "test",
                                     np.where(split_df["split"] == "valid", "valid", "train")))
        y = yolo.export_yolo(images, boxes, fold_split, WORK / "yolo_loso" / source)
        w = yolo.train(y, paths.outputs / "yolo_loso", model="yolo26n.pt", name=source)
        fold_valid = df[df["uid"].isin(fold_split.loc[fold_split["split"] == "valid", "uid"])]
        b = yolo.predict(w, pd.concat([fold_valid, te]))
        t = yolo.tune_conf(b[b["uid"].isin(fold_valid["uid"])], fold_valid)
        b = b[(b["conf"] >= t) & b["uid"].isin(te["uid"])]
        counted = with_counts(te, b)
        evaluate("B7 YOLO26n", "leave-one-session-out", source, counted, b)
        loso.append((counted, b))
    evaluate("B7 YOLO26n", "leave-one-session-out", "pooled",
             pd.concat([c for c, _ in loso]), pd.concat([b for _, b in loso]))

## Results table

In [ ]:
res = pd.DataFrame(results)
res.to_csv(paths.outputs / "results" / "baselines.csv", index=False)

cols = ["mae_total", "mae_live", "mae_dead", "viability_err_pp", "within_10pct", "r2_total",
        "precision_cell", "recall_cell", "f1_live", "f1_dead"]
summary = res[res["fold"].isin(["test", "pooled"])].set_index(["level", "method"])[cols]
summary.round(3)

## Diagnostics (random-split test)

In [ ]:
methods = list(preds)
fig, axes = plt.subplots(2, len(methods), figsize=(4 * len(methods), 8), squeeze=False)
for j, m in enumerate(methods):
    p = preds[m]
    for i, (t, q, label) in enumerate([("total_true", None, "Total"), ("dead_true", "dead_pred", "Dead")]):
        ax = axes[i, j]
        pred = p["live_pred"] + p["dead_pred"] if q is None else p[q]
        sns.scatterplot(x=p[t], y=pred, hue=p["source"], ax=ax, s=25, legend=(i == 0 and j == 0))
        lim = [0, max(p[t].max(), pred.max()) * 1.05]
        ax.plot(lim, lim, "k--", lw=0.8)
        ax.set(xlim=lim, ylim=lim, xlabel=f"true {label}", ylabel=f"predicted {label}", title=f"{m} · {label}")
fig.tight_layout()
fig.savefig(FIG / "01_pred_vs_true.png", dpi=150)

In [ ]:
by = []
for m, p in preds.items():
    for col, name in [("source", "session"), ("count_bin", "count bin")]:
        g = count_metrics_by(p, p[col])[["mae_total"]].reset_index(names="group")
        by.append(g.assign(method=m, by=name, group=g["group"].astype(str)))
by = pd.concat(by)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, ["session", "count bin"]):
    sns.barplot(data=by[by["by"] == name], x="group", y="mae_total", hue="method", ax=ax)
    ax.set(title=f"Total-count MAE by {name}", xlabel=name, ylabel="MAE (cells)")
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIG / "02_mae_by_session_and_count.png", dpi=150)

In [ ]:
lv = res[res["fold"].isin(["test", "pooled"])]
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=lv, x="method", y="mae_total", hue="level", ax=ax)
ax.set(title="Random split vs unseen session", xlabel="", ylabel="Total-count MAE (cells)")
fig.tight_layout()
fig.savefig(FIG / "03_random_vs_loso.png", dpi=150)

**Worst test images** for each detector (largest total-count error). Yellow: labelled boxes. Cyan / magenta: predicted Live / Dead.

In [ ]:
by_uid = df.set_index("uid", drop=False)
for m, pred_boxes, fname in [("B4 Cellpose-SAM", cp_boxes, "04_worst_cellpose.png"),
                             ("B7 YOLO26n", yolo_kept, "05_worst_yolo.png")]:
    p = preds[m].assign(err=lambda d: d["live_pred"] + d["dead_pred"] - d["total_true"])
    worst = p.reindex(p["err"].abs().sort_values(ascending=False).index).head(6)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, r in zip(axes.flat, worst.itertuples()):
        viz.show_prediction(by_uid.loc[r.uid], boxes, pred_boxes, ax=ax,
                            title=f"{r.uid} ({r.source[-3:]})  true {r.total_true}, pred {r.live_pred + r.dead_pred:.0f}")
    fig.suptitle(f"{m}: worst 6 test images")
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=120)

## Inference time per image

Measured on this Colab GPU, excluding model loading. Counting by hand takes roughly a minute or more per image.

In [ ]:
from pathlib import Path
import shutil
import tempfile

sample = test.head(20)
tmp = Path(tempfile.mkdtemp())
cellpose_zs.segment(sample.head(1), tmp)  # load the model once
t0 = time.perf_counter()
cellpose_zs.segment(sample.iloc[1:], tmp)
cp_sec = (time.perf_counter() - t0) / (len(sample) - 1)
shutil.rmtree(tmp)

yolo.predict(weights, sample.head(1))  # warm-up
t0 = time.perf_counter()
yolo.predict(weights, sample)
yolo_sec = (time.perf_counter() - t0) / len(sample)

timing = pd.Series({"B4 Cellpose-SAM": cp_sec, "B7 YOLO26n": yolo_sec}, name="seconds per image")
timing.round(3)

## Reference results (local run, M5 Pro)

Compare your Colab numbers with these; they should match closely apart from the timings and small YOLO training noise.

Full write-up with figures: `notes/baselines_results.md`.

| Random split (75 test images) | Total MAE | Within ±10% | R² | s / image |
|---|---|---|---|---|
| B0 training mean | 12.7 | 24% | −0.01 | — |
| B1 session mean | 6.9 | 45% | 0.71 | — |
| B4 Cellpose-SAM zero-shot | 5.6 | 60% | 0.76 | 1.8 |
| B7 YOLO26n fine-tuned | **1.6** | **96%** | **0.98** | **0.025** |

- **YOLO** is near the label-noise floor here: remaining errors are faint cells the annotators labelled inconsistently. Dead cells are its weak spot (F1 0.85 vs 0.96 for Live).
- **Cellpose** needs no training and keeps most of its accuracy on unseen sessions (MAE 5.6 → 6.3), but misses pale, out-of-focus cells. Its IoU-0.5 detection scores are low only because its outlines include the halo (0.89 precision and recall at IoU 0.3).
- **B1** confirms the session shortcut: R² 0.71 on the random split, −0.40 on unseen sessions.
- **Next:** YOLO leave-one-session-out (`RUN_YOLO_LOSO = True`), then the stress tests.